# Chapter 7 — Message Batches API · Homework (self-test)

**Goal:** check that you can decide when to batch, design `custom_id`s, triage results, resubmit only the failures with the right fix, and do the SLA and cost math.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 60–90 minutes
- Estimated API cost: **$0** (everything graded runs offline, on real SDK types); the one optional live cell costs < $0.001
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 10 exam-style questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises), the same bar as the Claude exam's 720/1000.

In [ ]:
import copy
import hashlib
import json
import math
import os
import re
from datetime import datetime, timedelta
from typing import Callable

import anthropic
from anthropic.types.messages import MessageBatchIndividualResponse
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, ValidationError

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
DEMO_MODEL = os.getenv("CLAUDE_BATCH_DEMO_MODEL", "claude-haiku-4-5")
client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") else None   # only the optional live cell uses it
print("anthropic", anthropic.__version__, "| MODEL =", MODEL)

The optional live cell needs a key. Skip this assert if you have none: everything graded is offline.

In [ ]:
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"

### Helpers (run, don't edit)

`make_result()` builds real `MessageBatchIndividualResponse` objects (the type `client.messages.batches.results()` yields), so code that passes these checks works on real results. `chunk_text()`, `PRICES`, `Invoice` and `EXTRACTION_SYSTEM` are shared by the exercises.

In [ ]:
# ---------- grading bookkeeping ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {exc}")


# ---------- shared definitions used by the exercises ----------
CUSTOM_ID_RE = re.compile(r"^[a-zA-Z0-9_-]{1,64}$")
FALLBACK_MODEL = "claude-haiku-4-5"
EXTRACTION_SYSTEM = ("Extract vendor, invoice_number, line_items_cents (integer cents) and total_cents as JSON. "
                     "Use null for anything not printed on the document.")


class Invoice(BaseModel):
    vendor: str
    invoice_number: str | None          # null = not printed (a valid answer)
    line_items_cents: list[int]
    total_cents: int | None             # null = not printed: the business needs it, so a human must look


# Standard prices, USD per million tokens (Claude API, October 2026). Batch = 50% of these.
PRICES = {
    "claude-sonnet-5-5": {"in": 2.00, "out": 10.00, "read_mult": 0.10, "min_cache": 512},
    "claude-haiku-4-5":  {"in": 1.00, "out": 5.00,  "read_mult": 0.10, "min_cache": 4_096},
    "claude-opus-5-5":   {"in": 4.00, "out": 20.00, "read_mult": 0.05, "min_cache": 512},
}
WRITE_MULT = {"5m": 1.25, "1h": 2.0}


def chunk_text(text: str, max_chars: int) -> list[str]:
    """Split on line boundaries into pieces of at most max_chars (very long lines are hard-split)."""
    chunks, current = [], ""
    for line in text.splitlines():
        while len(line) > max_chars:
            if current:
                chunks.append(current)
                current = ""
            chunks.append(line[:max_chars])
            line = line[max_chars:]
        candidate = f"{current}\n{line}" if current else line
        if len(candidate) > max_chars:
            chunks.append(current)
            current = line
        else:
            current = candidate
    return chunks + ([current] if current else [])


def make_result(custom_id: str, kind: str, *, text: str = "", stop_reason: str = "end_turn",
                error_type: str = "api_error", error_message: str = "") -> MessageBatchIndividualResponse:
    """Build a REAL SDK result object (the same type client.messages.batches.results() yields)."""
    if kind == "succeeded":
        result = {"type": "succeeded", "message": {
            "id": f"msg_{custom_id}", "type": "message", "role": "assistant", "model": MODEL,
            "content": [{"type": "text", "text": text}] if text else [], "stop_reason": stop_reason,
            "stop_sequence": None, "usage": {"input_tokens": 100, "output_tokens": 50},
            "stop_details": {"type": "refusal", "category": "cyber", "explanation": "declined"}
            if stop_reason == "refusal" else None}}
    elif kind == "errored":
        result = {"type": "errored", "error": {"type": "error", "error": {"type": error_type, "message": error_message}}}
    else:
        result = {"type": kind}                                   # "canceled" or "expired"
    return MessageBatchIndividualResponse.model_validate({"custom_id": custom_id, "result": result})


print("helpers ready")

## Part A — Quiz (10 questions)

Pick one letter per question. Domains: 4.5 batch processing, 5.3–5.5 reliability, cost and operations.

**Q1. (Batch monitoring)** Forty minutes after submitting a 5,000-request batch, `retrieve()` shows `processing_status: "in_progress"`, `request_counts.processing = 5000` and `succeeded = 0`. An engineer proposes canceling and rerunning everything synchronously. What is the best response?

- A) Cancel it: zero successes after 40 minutes means the batch is stuck.
- B) Keep polling with backoff: the outcome counters are filled only when the whole batch ends, and planning should assume up to 24 hours anyway.
- C) Submit a duplicate batch now and keep whichever finishes first.
- D) Download `results_url` now to inspect the partial results.

**Q2. (Crash recovery)** An orchestrator crashed right after `client.messages.batches.create()` returned. On restart the job submitted the same 20,000 requests again, and both batches were billed. Which change prevents this?

- A) Poll less often so the orchestrator is less likely to crash.
- B) Give each submission random `custom_id`s so duplicate batches can be detected afterwards.
- C) Persist the batch ID and its list of `custom_id`s immediately after `create()`; on restart, resume polling the saved batch (and use `batches.list()` to recover a lost ID) instead of resubmitting.
- D) Move the workload to the synchronous API, which is crash-safe.

**Q3. (Server tools in batches)** You enrich 20,000 company records with the web search server tool in a batch. A few results come back `succeeded` with `stop_reason: "pause_turn"`. What is the correct handling?

- A) Continue each one with a follow-up request (batch or synchronous): the original messages plus the paused assistant content, under the same `custom_id`.
- B) Treat them as failures and resubmit the original requests unchanged.
- C) Move the workload to the synchronous API, because server tools are not supported in batches.
- D) Double `max_tokens` and resubmit them, because the output was truncated.

**Q4. (custom_id design)** A team uses `uuid4()` as the `custom_id` of every request, including retries, and upserts results into the warehouse by `custom_id`. After two retry rounds the warehouse holds duplicate rows for the same contracts. What is the best fix?

- A) Deduplicate by the order of results after each batch.
- B) Encode the attempt number in the ID (`c-417-retry2`) so each attempt is traceable.
- C) Use the full file path of each contract as the `custom_id` so it is unique.
- D) Derive the `custom_id` deterministically from the contract's record key (hashing long or PII-bearing keys), keep it the same across retries with the attempt count in a ledger, and upsert by it.

**Q5. (SLA planning)** A compliance report must be in the regulator's portal by Monday 08:00. The plan assumes 24 hours of batch processing in the worst case plus 2 hours to validate the results and retry failures synchronously. What is the latest time to submit the batch?

- A) Monday 07:00, because most batches finish within an hour.
- B) Sunday 08:00, exactly 24 hours before the deadline.
- C) Sunday 06:00.
- D) Saturday 08:00, to leave room for a batch-based retry cycle.

**Q6. (Prompt caching)** A nightly batch on `claude-haiku-4-5` shares a 3,000-token system prompt marked with `cache_control`. On every result, both `cache_creation_input_tokens` and `cache_read_input_tokens` are 0. What is the most likely cause?

- A) Prompt caching does not work with the Message Batches API.
- B) The shared prefix is shorter than Haiku 4.5's minimum cacheable length (4,096 tokens), so it is silently not cached.
- C) Batch requests run out of order, so every request misses the cache.
- D) The 5-minute cache entries expired; switching to the 1-hour TTL will fix it.

**Q7. (Failure handling)** A batch ends with 40 errored requests: 30 `overloaded_error` and 10 `invalid_request_error` with "prompt is too long". What should the next batch contain?

- A) All 40 requests, unchanged.
- B) The entire original batch, because errors indicate a systemic problem.
- C) All 40 requests with a larger `max_tokens`.
- D) The 30 overloaded requests unchanged, plus chunked requests for the 10 oversized documents (IDs such as `doc-17-c0`, `doc-17-c1`), and nothing else.

**Q8. (Long outputs)** A team needs one response of about 250,000 tokens (a complete API reference) from `claude-sonnet-5-5`, generated in a single turn. Which approach works?

- A) The Message Batches API with the `output-300k-2026-03-24` beta header and `max_tokens` up to 300,000.
- B) The synchronous Messages API with streaming and `max_tokens: 250000`.
- C) A batch on `claude-haiku-4-5`, because it is the cheapest per output token.
- D) Any batch request: `max_tokens` has no upper limit in batches.

**Q9. (Data governance)** A customer with a Zero Data Retention (ZDR) agreement wants nightly summaries of support transcripts at the 50% batch price. What is the correct guidance?

- A) Use batches: ZDR applies automatically to every API endpoint.
- B) Use batches and delete each batch right after downloading results; that makes the workload ZDR-compliant.
- C) The Batches API is not ZDR-eligible (it stores inputs and outputs for up to 29 days); keep ZDR-covered data on the synchronous API unless the customer explicitly accepts batch retention for it.
- D) Use batches, but remove the system prompt so less data is stored.

**Q10. (Human review and metrics)** An extraction pipeline over 50,000 documents reports 97% of results accepted after validation. The team proposes cutting the human review sample from 5% to 0.5%. What should happen first?

- A) Approve it: 97% is above the 95% target.
- B) Break acceptance and sampled error rates down by document type and field, and keep stratified review for weak segments, before reducing review.
- C) Resubmit the 3% on a bigger model, then cut the review sample.
- D) Use larger batches so that the error rate averages out.

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365', 'q2': 'c85dcf1566dc5c0487a6fe1611c41ba32751e8080b552a419521a39c7fee21b2', 'q3': '2501e2fe43d858f1773d414ee14ab0a8cdf2afefbc4ffd20ae5d956e618f3ea3', 'q4': 'b634a8333266441f30307b2d3392d77b7b3bcf69bc053cabed2c65d2671fc12e', 'q5': 'e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f', 'q6': '621f4cab9e29d186b9dabeec79a433d2126c30a9e3f5712a1411fc67d73f1e40', 'q7': '45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457', 'q8': 'f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc', 'q9': 'e19608e84d37f3c94b5f88738c75a8c96772a5e61784e3e24b5512fa65ff63c7', 'q10': 'c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — Build a valid batch request list

Write `build_batch_requests(docs, model=MODEL, max_tokens=512)`. `docs` maps a **document key** (a record ID, a file path, an email, or a very long key) to its text. Return `(requests, id_to_key)`:

- one request per document: `{"custom_id": ..., "params": {...}}`;
- `custom_id` must match `^[a-zA-Z0-9_-]{1,64}$`, be **unique** and **deterministic** (same key → same ID on every run), and contain no paths or emails. A key that is already legal and at most 48 characters is used **as is**; anything else becomes `"d-" + sha256(key)[:24]`;
- `params`: `model`, `max_tokens`, `system=EXTRACTION_SYSTEM`, and one user message whose content is the text wrapped in `<document>...</document>`. No `stream`;
- `id_to_key` maps every `custom_id` back to the original key (your ledger).

Theory: [7.1](README.md#71-what-the-message-batches-api-is), [7.4](README.md#74-designing-custom_id)

In [ ]:
def build_batch_requests(docs: dict[str, str], model: str = MODEL, max_tokens: int = 512) -> tuple[list[dict], dict[str, str]]:
    """Return (requests, id_to_key). Rules in the task above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def check():
        docs = {
            "inv-2026-000417": "Invoice INV-417 ... TOTAL 120.00",
            "inv 2026 000417": "A different document whose key differs only by spaces",
            "contracts/2026/ACME Corp/MSA v3.pdf": "Master services agreement ...",
            "jane.doe@example.com": "Receipt emailed by a customer ...",
            "customer_" + "x" * 80: "A record with a very long natural key",
        }
        out = build_batch_requests(docs)
        assert isinstance(out, tuple) and len(out) == 2, "return a tuple (requests, id_to_key)"
        requests, id_to_key = out
        assert isinstance(requests, list) and len(requests) == len(docs), f"one request per document ({len(docs)})"
        ids = [r.get("custom_id") for r in requests]
        bad = [i for i in ids if not isinstance(i, str) or not CUSTOM_ID_RE.match(i)]
        assert not bad, f"illegal custom_id(s) {bad}: must match ^[a-zA-Z0-9_-]{{1,64}}$"
        assert len(set(ids)) == len(ids), ("custom_ids collide. 'inv 2026 000417' must not become the same id as "
                                           "'inv-2026-000417': hash keys that are not already legal")
        assert not any(("@" in i or "/" in i) for i in ids), "no emails or paths in custom_id (PII, illegal chars)"
        assert "inv-2026-000417" in ids, "a key that is already legal and short should be used as is (readable in ops)"
        assert isinstance(id_to_key, dict) and sorted(id_to_key.values()) == sorted(docs), \
            "id_to_key must map every custom_id back to its original document key"
        assert all(id_to_key[r["custom_id"]] in docs for r in requests), "id_to_key[custom_id] must be the original key"
        again, _ = build_batch_requests(docs)
        assert [r["custom_id"] for r in again] == ids, "custom_ids must be deterministic (same key -> same id on every run)"
        for r in requests:
            p = r.get("params", {})
            key = id_to_key[r["custom_id"]]
            assert p.get("model") == MODEL and p.get("max_tokens", 0) >= 1, "params need model and max_tokens >= 1"
            assert p.get("system") == EXTRACTION_SYSTEM, "use EXTRACTION_SYSTEM as the system prompt"
            assert "stream" not in p, "stream is rejected inside batch requests"
            msgs = p.get("messages", [])
            assert len(msgs) == 1 and msgs[0].get("role") == "user", "one user message per request"
            content = msgs[0].get("content", "")
            assert docs[key] in content and "<document>" in content, \
                "the user content must contain the document text wrapped in <document> tags"
    _run_check(1, check)


check_ex1()

### Exercise 2 — Batch or synchronous?

Write `choose_api(w, buffer_hours=2.0)` for a workload dict with optional keys `blocking`, `sla_hours` (`None` = no deadline), `zdr_required`, `needs_client_tools_midrequest`, `platform` (`"claude-api"`, `"bedrock"`, `"vertex"`, `"foundry"`). Apply the rules **in this order**:

1. `blocking` → `"sync"`
2. `zdr_required` → `"sync"`
3. `sla_hours` is set and `sla_hours < 24 + buffer_hours` → `"sync"`
4. `needs_client_tools_midrequest` → `"sync"`
5. `platform` is `"bedrock"` or `"vertex"` → `"provider_batch"` (Bedrock batch inference, Vertex AI batch predictions)
6. `platform` is `"foundry"` → `"sync"` (Microsoft Foundry documents no batch option for Claude)
7. otherwise → `"batch"`

Other keys (`volume`, `uses_server_tools`, ...) must **not** change the answer.

Theory: [7.2](README.md#72-batch-vs-synchronous-matching-the-api-to-latency-requirements), [7.11](README.md#711-operations-rate-limits-retention-privacy-and-platforms)

In [ ]:
def choose_api(w: dict, buffer_hours: float = 2.0) -> str:
    """Return 'sync', 'batch' or 'provider_batch'. Rules in the task above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    def check():
        cases = [
            ({"name": "pre-merge check", "blocking": True, "volume": 50_000}, "sync",
             "a blocking pre-merge check stays synchronous, whatever the volume"),
            ({"name": "overnight report", "blocking": False, "sla_hours": 36}, "batch", "36 h SLA fits 24 h + buffer"),
            ({"name": "weekly audit", "blocking": False, "sla_hours": None}, "batch", "no deadline: batch it"),
            ({"name": "triage, 4 h SLA", "blocking": False, "sla_hours": 4, "volume": 100_000}, "sync",
             "a 4 h SLA cannot absorb the 24 h worst case (most batches finishing in 1 h is not a guarantee)"),
            ({"name": "edge: 26 h SLA", "blocking": False, "sla_hours": 26}, "batch", "26 >= 24 + 2: just fits"),
            ({"name": "edge: 25.5 h SLA", "blocking": False, "sla_hours": 25.5}, "sync", "25.5 < 24 + 2"),
            ({"name": "ZDR customer", "blocking": False, "sla_hours": 48, "zdr_required": True}, "sync",
             "the Batches API is not ZDR-eligible"),
            ({"name": "read_file review loop", "blocking": False, "sla_hours": 72,
              "needs_client_tools_midrequest": True}, "sync", "a batch cannot run your client tool and continue"),
            ({"name": "web-search enrichment", "blocking": False, "sla_hours": 72, "uses_server_tools": True}, "batch",
             "server tools run inside the batch: batch it"),
            ({"name": "nightly on Bedrock", "blocking": False, "sla_hours": 48, "platform": "bedrock"},
             "provider_batch", "/v1/messages/batches is not supported on Bedrock: use the provider's batch service"),
            ({"name": "nightly on Vertex", "blocking": False, "sla_hours": 48, "platform": "vertex"},
             "provider_batch", "/v1/messages/batches is not supported on Vertex AI"),
            ({"name": "nightly on Foundry", "blocking": False, "sla_hours": 48, "platform": "foundry"},
             "sync", "Microsoft Foundry documents no batch option for Claude: synchronous calls behind your own queue"),
            ({"name": "nightly on Claude API", "blocking": False, "sla_hours": 48, "platform": "claude-api"},
             "batch", "first-party API: Message Batches"),
        ]
        for w, expected, why in cases:
            got = choose_api(w)
            assert got == expected, f"{w['name']}: got {got!r}, expected {expected!r} ({why})"
        got = choose_api({"name": "x", "blocking": False, "sla_hours": 30}, buffer_hours=8)
        assert got == "sync", f"buffer_hours=8 with a 30 h SLA: 30 < 24 + 8, expected 'sync', got {got!r}"
    _run_check(2, check)


check_ex2()

### Exercise 3 — Triage batch results

Write `triage(results, schema=Invoice)`. `results` is an iterable of real SDK `MessageBatchIndividualResponse` objects (exactly what `client.messages.batches.results()` yields; the checker builds them with `make_result`). Return a dict with the buckets `accept`, `retry_as_is`, `fix_then_retry`, `fallback_model`, `human`, each a list of `custom_id`s; every ID lands in exactly one bucket.

| Result | Bucket |
|---|---|
| `canceled` or `expired` | `retry_as_is` |
| `errored` with `invalid_request_error` | `fix_then_retry` |
| `errored`, any other error type | `retry_as_is` |
| succeeded, `stop_reason == "refusal"` | `fallback_model` |
| succeeded, `stop_reason == "max_tokens"` or `"model_context_window_exceeded"` (both mean truncated) | `fix_then_retry` |
| succeeded, any other non-`end_turn` stop | `human` |
| `end_turn`, text fails `schema.model_validate_json` | `fix_then_retry` |
| `end_turn`, valid, `total_cents is None` | `human` (the value is not in the document) |
| `end_turn`, valid, `sum(line_items_cents) != total_cents` | `fix_then_retry` |
| `end_turn`, valid and consistent (`invoice_number` may be `None`) | `accept` |

Hint: for errored results the error type is at `item.result.error.error.type`.

Theory: [7.6](README.md#76-result-types-and-resubmitting-only-the-failures)

In [ ]:
def triage(results, schema: type[BaseModel] = Invoice) -> dict[str, list[str]]:
    """Sort every result's custom_id into exactly one bucket. Rules in the task above."""
    buckets = {"accept": [], "retry_as_is": [], "fix_then_retry": [], "fallback_model": [], "human": []}
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        good = '{"vendor": "Acme", "invoice_number": "INV-1", "line_items_cents": [1000, 500], "total_cents": 1500}'
        cases = [
            (make_result("ok", "succeeded", text=good), "accept", "valid JSON whose lines add up"),
            (make_result("no-number", "succeeded", text=good.replace('"INV-1"', "null")), "accept",
             "invoice_number null means 'not printed': a valid answer, not a failure"),
            (make_result("no-total", "succeeded", text=good.replace("1500}", "null}")), "human",
             "total_cents null: the information is absent, a retry cannot create it"),
            (make_result("bad-sum", "succeeded", text=good.replace("1500}", "1700}")), "fix_then_retry",
             "lines do not add up to the total: retry with specific feedback"),
            (make_result("bad-schema", "succeeded", text='{"vendor": "Acme", "total_cents": "15.00"}'), "fix_then_retry",
             "schema-invalid output: retry with the validation error"),
            (make_result("truncated", "succeeded", text='{"vendor": "Ac', stop_reason="max_tokens"), "fix_then_retry",
             "stop_reason max_tokens: raise max_tokens"),
            (make_result("ctx-full", "succeeded", text='{"vendor": "Ac', stop_reason="model_context_window_exceeded"),
             "fix_then_retry", "stop_reason model_context_window_exceeded is a truncation: chunk the input"),
            (make_result("refused", "succeeded", stop_reason="refusal"), "fallback_model",
             "a refusal arrives as succeeded + stop_reason refusal: route to a fallback model"),
            (make_result("tool-stop", "succeeded", text="I need read_file", stop_reason="tool_use"), "human",
             "a client tool_use stop cannot be finished inside a batch: redesign"),
            (make_result("too-long", "errored", error_type="invalid_request_error",
                         error_message="prompt is too long: 1204331 tokens > 1000000 maximum"), "fix_then_retry",
             "invalid_request_error is deterministic: fix (chunk) before resending"),
            (make_result("overloaded", "errored", error_type="overloaded_error", error_message="Overloaded"),
             "retry_as_is", "overloaded_error is transient: resend unchanged"),
            (make_result("expired", "expired"), "retry_as_is", "expired never ran: resend"),
            (make_result("canceled", "canceled"), "retry_as_is", "canceled never ran: resend if still needed"),
        ]
        shuffled = [cases[i] for i in (5, 0, 12, 3, 8, 1, 11, 6, 2, 10, 7, 4, 9)]   # results are unordered
        out = triage([r for r, _, _ in shuffled])
        assert isinstance(out, dict), "return the buckets dict"
        expected_keys = {"accept", "retry_as_is", "fix_then_retry", "fallback_model", "human"}
        assert set(out) == expected_keys, f"bucket names must be exactly {sorted(expected_keys)}"
        placed = [cid for ids in out.values() for cid in ids]
        assert sorted(placed) == sorted(r.custom_id for r, _, _ in cases), \
            "every custom_id must appear in exactly one bucket (no drops, no duplicates)"
        where = {cid: b for b, ids in out.items() for cid in ids}
        for r, expected, why in cases:
            assert where[r.custom_id] == expected, f"{r.custom_id}: put in {where[r.custom_id]!r}, expected {expected!r} ({why})"
    _run_check(3, check)


check_ex3()

### Exercise 4 — SLA windows and deadlines

Write two functions:

- `submission_window(sla_hours, processing_hours=24, buffer_hours=2, batch_retry_cycles=0)`: the longest window `W` with `W + processing_hours × (1 + batch_retry_cycles) + buffer_hours ≤ sla_hours`. If `W ≤ 0`, raise `ValueError` (batches cannot meet that SLA).
- `latest_submission(deadline, processing_hours=24, buffer_hours=2)`: the latest `datetime` at which a one-off batch can be submitted and still be processed and validated by `deadline`.

Theory: [7.8](README.md#78-sla-planning-and-submission-cadence)

In [ ]:
def submission_window(sla_hours: float, processing_hours: float = 24, buffer_hours: float = 2,
                      batch_retry_cycles: int = 0) -> float:
    """Longest window between submissions that meets the SLA in the worst case. Raise ValueError if none."""
    # TODO
    raise NotImplementedError


def latest_submission(deadline: datetime, processing_hours: float = 24, buffer_hours: float = 2) -> datetime:
    """Latest time to submit a one-off batch that must be done (and validated) by `deadline`."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        cases = [((30,), {}, 4, "30 h SLA - 24 h processing - 2 h buffer = 4 h windows"),
                 ((36,), {}, 10, "36 - 24 - 2"),
                 ((30,), {"buffer_hours": 0}, 6, "with no buffer, 6 h windows exactly hit 30 h"),
                 ((54,), {"batch_retry_cycles": 1}, 4, "one batch-based retry adds another 24 h: 54 - 48 - 2"),
                 ((40,), {"processing_hours": 12}, 26, "use the processing_hours you are given")]
        for args, kw, expected, why in cases:
            got = submission_window(*args, **kw)
            assert isinstance(got, (int, float)) and abs(got - expected) < 1e-9, \
                f"submission_window({args[0]}, **{kw}) = {got!r}, expected {expected} ({why})"
        for args, kw, why in [((26,), {}, "26 - 24 - 2 = 0: no room at all"),
                              ((30,), {"batch_retry_cycles": 1}, "a batch retry cycle cannot fit in 30 h"),
                              ((20,), {}, "the SLA is shorter than the 24 h worst case")]:
            try:
                got = submission_window(*args, **kw)
            except ValueError:
                continue
            raise AssertionError(f"submission_window({args[0]}, **{kw}) returned {got!r}; raise ValueError ({why})")
        monday = datetime(2026, 10, 5, 8, 0)
        got = latest_submission(monday)
        assert got == datetime(2026, 10, 4, 6, 0), f"Monday 08:00 deadline: expected Sunday 06:00, got {got}"
        got = latest_submission(monday, buffer_hours=6)
        assert got == datetime(2026, 10, 4, 2, 0), f"with a 6 h buffer expected Sunday 02:00, got {got}"
    _run_check(4, check)


check_ex4()

### Exercise 5 — Estimate batch cost with caching

Write `estimate_batch_cost(n_docs, prefix_tokens, doc_tokens, output_tokens, model="claude-sonnet-5-5", cache_hit_rate=0.0, ttl="5m")` returning total USD, using `PRICES` and `WRITE_MULT` from the helpers:

- every token type costs **50%** of the standard price (batch discount);
- per request: a shared prefix of `prefix_tokens`, then `doc_tokens` of unique input, then `output_tokens` of output;
- if `cache_hit_rate > 0` **and** `prefix_tokens >= PRICES[model]["min_cache"]`: a fraction `cache_hit_rate` of the prefix tokens are cache **reads** (`read_mult` × batch input price) and the rest are cache **writes** (`WRITE_MULT[ttl]` × batch input price);
- otherwise the prefix is billed as normal batch input (a prefix below the minimum is silently not cached).

Theory: [7.9](README.md#79-cost-modeling-batch-discount-prompt-caching-and-model-choice)

In [ ]:
def estimate_batch_cost(n_docs: int, prefix_tokens: int, doc_tokens: int, output_tokens: int,
                        model: str = "claude-sonnet-5-5", cache_hit_rate: float = 0.0, ttl: str = "5m") -> float:
    """Total USD for a batch run. Rules in the task above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        base = dict(n_docs=10_000, prefix_tokens=3_000, doc_tokens=2_000, output_tokens=500)
        cases = [({}, 75.00, "Sonnet 5.5 batch, no caching: half of the $150 synchronous price"),
                 ({"cache_hit_rate": 0.7}, 58.35, "70% reads at 0.1x and 30% writes at 1.25x of the BATCH input price"),
                 ({"cache_hit_rate": 0.95, "ttl": "1h"}, 50.85, "1-hour writes cost 2x, not 1.25x"),
                 ({"model": "claude-haiku-4-5", "cache_hit_rate": 0.7}, 37.50,
                  "3,000 tokens is below Haiku 4.5's 4,096-token minimum: nothing is cached"),
                 ({"model": "claude-opus-5-5", "cache_hit_rate": 0.7}, 114.60,
                  "Opus 5.5 cache reads are 0.05x (use read_mult from PRICES)"),
                 ({"model": "claude-haiku-4-5", "prefix_tokens": 5_000, "cache_hit_rate": 0.7}, 33.625,
                  "a 5,000-token prefix IS cacheable on Haiku 4.5")]
        for kw, expected, why in cases:
            args = {**base, **kw}
            got = estimate_batch_cost(**args)
            assert isinstance(got, (int, float)), f"return a number, got {got!r}"
            assert abs(got - expected) < 0.01, f"{kw or 'defaults'}: got ${got:.3f}, expected ${expected:.3f} ({why})"
    _run_check(5, check)


check_ex5()

### Exercise 6 — Resubmit only the failures, with the right fix

Write `build_resubmission(failures, ledger, max_attempts=3, chunk_chars=4000)`.

- `failures` maps `custom_id` → a failure code: `expired`, `canceled`, `overloaded`, `max_tokens`, `refusal`, `schema`, `prompt_too_long` or `not_found`.
- `ledger` maps every `custom_id` (successes included) → `{"text", "attempt", "answer", "params"}`, where `params` is the original request params.

Return `(requests, human_ids)`:

| Condition | Action |
|---|---|
| `attempt >= max_attempts`, or code `not_found` | add to `human_ids`, do not resubmit |
| `expired`, `canceled`, `overloaded` | same `custom_id`, params **unchanged** |
| `max_tokens` | double `max_tokens` |
| `refusal` | `model = FALLBACK_MODEL` |
| `schema` | append `{"role": "assistant", "content": answer}` and a user message containing the word **feedback** |
| `prompt_too_long` | one request per piece of `chunk_text(text, chunk_chars)`, IDs `f"{custom_id}-c{i}"`, each with a single user message `<document>\n{piece}\n</document>` and the other params unchanged |

Never include successes, and **never mutate the ledger** (copy params before changing them).

Theory: [7.6](README.md#76-result-types-and-resubmitting-only-the-failures), [7.12](README.md#712-reliability-around-a-batch-pipeline-domain-5)

In [ ]:
def build_resubmission(failures: dict[str, str], ledger: dict[str, dict], max_attempts: int = 3,
                       chunk_chars: int = 4_000) -> tuple[list[dict], list[str]]:
    """Return (requests for a new batch with ONLY the fixable failures, custom_ids for human review)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    def check():
        def entry(text, attempt=1, answer=None):
            return {"text": text, "attempt": attempt, "answer": answer, "params": {
                "model": MODEL, "max_tokens": 512, "system": EXTRACTION_SYSTEM,
                "messages": [{"role": "user", "content": f"<document>\n{text}\n</document>"}]}}

        long_text = "\n".join(f"- line item {i:04d} ........ {i}.00" for i in range(700))   # ~23,000 chars
        ledger = {
            "ok-1": entry("never failed"),
            "exp-1": entry("expired doc"), "can-1": entry("canceled doc"), "ovl-1": entry("overloaded doc"),
            "trunc-1": entry("very long output doc"), "ref-1": entry("pen-test invoice"),
            "schema-1": entry("doc", answer='{"vendor": "Acme", "total_cents": "15.00"}'),
            "long-1": entry(long_text),
            "nf-1": entry("doc without a total"),
            "tired-1": entry("failed three times", attempt=3),
        }
        failures = {"exp-1": "expired", "can-1": "canceled", "ovl-1": "overloaded", "trunc-1": "max_tokens",
                    "ref-1": "refusal", "schema-1": "schema", "long-1": "prompt_too_long", "nf-1": "not_found",
                    "tired-1": "overloaded"}
        before = copy.deepcopy(ledger)
        out = build_resubmission(failures, ledger)
        assert isinstance(out, tuple) and len(out) == 2, "return (requests, human_ids)"
        requests, human = out
        assert ledger == before, "do not mutate the ledger's params (copy them before changing max_tokens/model/messages)"
        by_id = {r["custom_id"]: r["params"] for r in requests}
        assert len(by_id) == len(requests), "custom_ids in the new batch must be unique"
        assert "ok-1" not in by_id, "never resubmit successes"
        assert sorted(human) == ["nf-1", "tired-1"], \
            f"human review should be ['nf-1', 'tired-1'] (missing info; attempts exhausted), got {sorted(human)}"
        assert "nf-1" not in by_id and "tired-1" not in by_id, "items routed to humans must not also be resubmitted"
        for cid in ("exp-1", "can-1", "ovl-1"):
            assert by_id.get(cid) == ledger[cid]["params"], f"{cid}: transient/never-ran failures go back UNCHANGED"
        assert by_id.get("trunc-1", {}).get("max_tokens") == 1024, "max_tokens truncation: double max_tokens (512 -> 1024)"
        assert by_id.get("ref-1", {}).get("model") == FALLBACK_MODEL, "refusal: switch model to FALLBACK_MODEL"
        msgs = by_id.get("schema-1", {}).get("messages", [])
        assert len(msgs) == 3 and msgs[1]["role"] == "assistant" and msgs[2]["role"] == "user", \
            "schema: append the previous answer (assistant) and a feedback message (user)"
        assert msgs[1]["content"] == ledger["schema-1"]["answer"] and "feedback" in msgs[2]["content"].lower(), \
            "the assistant turn is the ledger's answer; the user turn must contain the word 'feedback'"
        assert "long-1" not in by_id, "an oversized document must be split, not resent whole"
        parts = chunk_text(long_text, 4_000)
        chunk_ids = [f"long-1-c{i}" for i in range(len(parts))]
        assert [c for c in by_id if c.startswith("long-1")] == chunk_ids, \
            f"expected chunk ids {chunk_ids[:2]}... (custom_id + '-c' + index, one per chunk_text() piece)"
        for cid, part in zip(chunk_ids, parts):
            content = by_id[cid]["messages"][-1]["content"]
            assert part in content and "<document>" in content, f"{cid}: content must hold its chunk in <document> tags"
            assert by_id[cid]["model"] == MODEL and by_id[cid]["system"] == EXTRACTION_SYSTEM, \
                f"{cid}: keep the other params (model, system) from the ledger"
        expected_ids = {"exp-1", "can-1", "ovl-1", "trunc-1", "ref-1", "schema-1", *chunk_ids}
        assert set(by_id) == expected_ids, f"unexpected request ids: {sorted(set(by_id) ^ expected_ids)}"
    _run_check(6, check)


check_ex6()

### Exercise 1b — your requests against the real API (live, optional, ~$0.001, not scored)

Run it once Exercise 1 passes. It submits a 2-request batch on Claude Haiku 4.5, shows that the API accepted your `custom_id`s, and cancels the batch.

In [ ]:
# (live, optional, ~$0.001, not scored) submit two requests built by YOUR build_batch_requests, then clean up.
if not RESULTS.get(1):
    print("Finish Exercise 1 first (its checker must pass), then run this cell.")
else:
    reqs, id_to_key = build_batch_requests({"inv-hw-001": "Invoice INV-1, Acme, TOTAL 12.00 EUR",
                                            "jane@example.com": "Receipt from Globex, Amount paid 3.50 EUR"},
                                           model=DEMO_MODEL, max_tokens=100)
    batch = client.messages.batches.create(requests=reqs)
    print(batch.id, batch.processing_status, batch.request_counts)
    canceled = client.messages.batches.cancel(batch.id)    # we only wanted to see it accepted
    print("after cancel:", canceled.processing_status, "| custom_ids sent:", list(id_to_key))

## Part C — Architecture scenario

**Nightly contract extraction.** A legal-ops team receives about **50,000 contracts per night** (PDFs, 2–200 pages; some over 1M tokens). For each contract they need 25 fields (parties, dates, renewal terms, liability caps) in a review tool by **30 hours after arrival**. During the day, lawyers also ask ad-hoc questions about single contracts and expect answers in seconds. Budget matters: finance asked for the cheapest design that meets the SLA. The customer is on the Claude API (no ZDR agreement) and requires an audit trail of every extracted value.

Describe your design (15–25 lines):
1. Which parts use the Message Batches API and which use the synchronous API, and why.
2. `custom_id` design, the manifest/ledger, and crash recovery.
3. Pre-flight before the first full run, and how you handle contracts that do not fit the context window.
4. Submission cadence for the 30-hour SLA, and how retries fit inside it.
5. Result handling: result types, `stop_reason`s, validation, what gets resubmitted (and how), what goes to humans.
6. Cost levers and how you will measure actual cost.
7. Operations: rate limits, retention, monitoring and reporting.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **API split:** nightly extraction on the **Message Batches API** (latency-tolerant, 50% off); lawyers' ad-hoc questions on the **synchronous** API (someone is blocked). No "batch with a sync fallback" for the interactive path.
- **`custom_id`:** derived from the contract's record key (hashed if long or PII-bearing), `-cN` suffixes for chunks, stable across retries. A **ledger** keyed by `custom_id` with source path, content hash, attempt, batch IDs, status. The batch ID is persisted right after `create()`; restarts resume polling (and `batches.list()` recovers lost IDs).
- **Pre-flight:** structured outputs (`output_config.format`) for the 25-field schema with nullable fields for absent data; a **stratified sample** (by contract type) run synchronously to refine the prompt and measure first-pass success by type and field; one synchronous dry run of the exact request shape; `count_tokens` to find oversized contracts and **chunk them before submission** (or route them to a different strategy), large PDFs via the Files API (`file_id`) to stay under 256 MB per batch; split 50,000 into several batches for manageability and queue limits.
- **Cadence:** worst case = W + 24 h + buffer ≤ 30 h, so **~4-hour submission windows** with a 2-hour buffer (or a fixed nightly cut-off computed the same way). Retries that must land inside the SLA go **synchronous**; batch-priced retries would need SLA ≥ W + 48 h + buffer.
- **Result handling:** stream results and match by `custom_id`, never by position. Four result types; `succeeded` checked for `refusal` (fallback model), `max_tokens` (raise the cap or narrow the task), `pause_turn`, schema and semantic validation (dates consistent, renewal after start). Resubmit **only** failed IDs with the matching modification; `invalid_request_error` fixed before resending; transient errors and `expired` resent as is; attempt cap. Fields genuinely absent stay `null` and go to **human review**, not retry. Reconciliation: every manifest ID ends in exactly one terminal state.
- **Cost:** batch × prompt caching (identical prefix: instructions + schema + few-shot first, 1-hour TTL, warm with a one-request batch) × model tiering (test a smaller model on the sample; check the minimum cacheable length). Measure real cost from `usage` (including cache fields) per batch and per document.
- **Operations:** batch RPM and queue limits (poll with backoff), workspace scoping, **29-day retention** (delete batches after ingestion if policy requires; note batches are not ZDR-eligible), coverage report per run **by contract type and field**, stratified human sampling before reducing review, idempotent upserts keyed by `custom_id`, and an audit trail linking each value to its source, model, attempt and batch.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_ex = 6
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (10 + n_ex)
print(f"Quiz:       {quiz}/10 ({quiz / 10:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to Chapter 8" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Next chapter: [Chapter 8 — Task Decomposition Strategies](../08-task-decomposition/README.md)

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **B.** While a batch is `in_progress`, `succeeded`, `errored`, `canceled` and `expired` all stay 0; they are filled when the batch ends. There is no latency SLA, so 40 minutes means nothing. A wastes the work done so far, C pays twice, and D fails: `results_url` is only set after the batch ends.

**Q2:** **C.** Persisting the batch ID (a manifest) right after `create()` makes a restart resume instead of resubmit (Domain 5.4). A doesn't address the cause. B makes duplicates *harder* to reconcile, not impossible to create. D gives up the discount; synchronous calls are not inherently crash-safe either.

**Q3:** **A.** Server tools run inside batches. `pause_turn` means the server-side loop hit its per-turn iteration limit; you continue by re-sending the paused assistant content in a follow-up request. B restarts the work from zero, C is false, and D confuses `pause_turn` with `max_tokens`.

**Q4:** **D.** One logical item = one stable key, so retries upsert onto the same row and audits can trace every attempt in the ledger. A relies on an order the API does not guarantee. B creates a new key per attempt (the same duplicate problem). C puts illegal characters and possibly PII into the ID.

**Q5:** **C.** Latest submit = deadline − 24 h − 2 h = Sunday 06:00. A plans from the typical hour, not the bound. B leaves no time for validation and retries. D is earlier than needed (the plan retries synchronously), and it would not even fit a second 24-hour cycle plus the 2-hour buffer (that needs Saturday 06:00).

**Q6:** **B.** Haiku 4.5's minimum cacheable prompt is 4,096 tokens; shorter prefixes are processed without caching and no error is returned (both cache fields stay 0). A is false (caching and batches stack). C would lower the hit rate, not zero the *creation* tokens. D: TTL doesn't explain why nothing was ever written.

**Q7:** **D.** Resubmit only the failures, each with the fix for its cause: transient overloads unchanged, oversized documents chunked with derived `custom_id`s. A resends 10 deterministic failures. B reprocesses successes (double cost, duplicates). C changes the output cap, but the problem is input size.

**Q8:** **A.** Extended output (beta) raises `max_tokens` to 300K only for batch requests on supported models, including Sonnet 5.5; the synchronous cap stays at 128K (B). Haiku 4.5 is not a supported model (C). D is false: 300K is the cap with the beta.

**Q9:** **C.** Batch processing stores request and response data for up to 29 days, and the feature is marked not ZDR-eligible. Deleting early (B) reduces exposure but does not make it eligible. A and D misread how ZDR works.

**Q10:** **B.** An aggregate can hide a document type or field with a much higher error rate. Measure by segment (stratified sampling) before reducing review (Domain 5.5). A trusts the aggregate. C and D don't measure anything.
</details>

<details><summary>Exercise 1 — Build a valid batch request list: reference solution</summary>

```python
def make_custom_id(key: str) -> str:
    if CUSTOM_ID_RE.match(key) and len(key) <= 48:
        return key                                                   # clean keys stay readable
    return "d-" + hashlib.sha256(key.encode()).hexdigest()[:24]       # paths, emails, long keys: hashed


def build_batch_requests(docs: dict[str, str], model: str = MODEL, max_tokens: int = 512) -> tuple[list[dict], dict[str, str]]:
    requests, id_to_key = [], {}
    for key, text in docs.items():
        cid = make_custom_id(key)
        if cid in id_to_key:
            raise ValueError(f"custom_id collision: {key!r} and {id_to_key[cid]!r}")
        id_to_key[cid] = key
        requests.append({"custom_id": cid, "params": {
            "model": model, "max_tokens": max_tokens, "system": EXTRACTION_SYSTEM,
            "messages": [{"role": "user", "content": f"<document>\n{text}\n</document>"}],
        }})
    return requests, id_to_key
```
</details>

<details><summary>Exercise 2 — Batch or synchronous?: reference solution</summary>

```python
def choose_api(w: dict, buffer_hours: float = 2.0) -> str:
    if w.get("blocking"):
        return "sync"                                   # someone is waiting; no latency SLA on batches
    if w.get("zdr_required"):
        return "sync"                                   # Batches API is not ZDR-eligible
    sla = w.get("sla_hours")
    if sla is not None and sla < 24 + buffer_hours:
        return "sync"                                   # plan from the 24 h bound, not the typical hour
    if w.get("needs_client_tools_midrequest"):
        return "sync"                                   # a batch can't run your tools and continue
    if w.get("platform") in ("bedrock", "vertex"):
        return "provider_batch"                         # /v1/messages/batches is not supported there
    if w.get("platform") == "foundry":
        return "sync"                                   # no documented batch option for Claude on Foundry
    return "batch"
```
</details>

<details><summary>Exercise 3 — Triage batch results: reference solution</summary>

```python
def triage(results, schema: type[BaseModel] = Invoice) -> dict[str, list[str]]:
    buckets = {"accept": [], "retry_as_is": [], "fix_then_retry": [], "fallback_model": [], "human": []}
    for item in results:
        cid, r = item.custom_id, item.result
        if r.type in ("canceled", "expired"):
            buckets["retry_as_is"].append(cid)
        elif r.type == "errored":
            kind = "fix_then_retry" if r.error.error.type == "invalid_request_error" else "retry_as_is"
            buckets[kind].append(cid)
        elif r.message.stop_reason == "refusal":
            buckets["fallback_model"].append(cid)
        elif r.message.stop_reason in ("max_tokens", "model_context_window_exceeded"):
            buckets["fix_then_retry"].append(cid)            # truncated: raise max_tokens or chunk the input
        elif r.message.stop_reason != "end_turn":
            buckets["human"].append(cid)
        else:
            text = "".join(b.text for b in r.message.content if b.type == "text")
            try:
                inv = schema.model_validate_json(text)
            except ValidationError:
                buckets["fix_then_retry"].append(cid)
                continue
            if inv.total_cents is None:
                buckets["human"].append(cid)                 # missing in the source: a retry can't create it
            elif sum(inv.line_items_cents) != inv.total_cents:
                buckets["fix_then_retry"].append(cid)        # semantic error: retry with specific feedback
            else:
                buckets["accept"].append(cid)
    return buckets
```
</details>

<details><summary>Exercise 4 — SLA windows and deadlines: reference solution</summary>

```python
def submission_window(sla_hours: float, processing_hours: float = 24, buffer_hours: float = 2,
                      batch_retry_cycles: int = 0) -> float:
    window = sla_hours - processing_hours * (1 + batch_retry_cycles) - buffer_hours
    if window <= 0:
        raise ValueError(f"SLA {sla_hours} h cannot absorb {1 + batch_retry_cycles} batch cycle(s) + buffer; "
                         "use synchronous calls for (part of) the work")
    return window


def latest_submission(deadline: datetime, processing_hours: float = 24, buffer_hours: float = 2) -> datetime:
    return deadline - timedelta(hours=processing_hours + buffer_hours)
```
</details>

<details><summary>Exercise 5 — Estimate batch cost with caching: reference solution</summary>

```python
def estimate_batch_cost(n_docs: int, prefix_tokens: int, doc_tokens: int, output_tokens: int,
                        model: str = "claude-sonnet-5-5", cache_hit_rate: float = 0.0, ttl: str = "5m") -> float:
    p = PRICES[model]
    batch_in, batch_out = p["in"] * 0.5 / 1e6, p["out"] * 0.5 / 1e6        # 50% off every token type
    if cache_hit_rate > 0 and prefix_tokens >= p["min_cache"]:
        reads = n_docs * cache_hit_rate * prefix_tokens
        writes = n_docs * (1 - cache_hit_rate) * prefix_tokens
        prefix_cost = reads * batch_in * p["read_mult"] + writes * batch_in * WRITE_MULT[ttl]
    else:                                                                   # too short to cache: silently full price
        prefix_cost = n_docs * prefix_tokens * batch_in
    return prefix_cost + n_docs * doc_tokens * batch_in + n_docs * output_tokens * batch_out
```
</details>

<details><summary>Exercise 6 — Resubmit only the failures, with the right fix: reference solution</summary>

```python
def build_resubmission(failures: dict[str, str], ledger: dict[str, dict], max_attempts: int = 3,
                       chunk_chars: int = 4_000) -> tuple[list[dict], list[str]]:
    requests, human = [], []
    for cid, code in failures.items():
        entry = ledger[cid]
        params = copy.deepcopy(entry["params"])                  # never mutate the ledger
        if code == "not_found" or entry["attempt"] >= max_attempts:
            human.append(cid)
        elif code in ("expired", "canceled", "overloaded"):
            requests.append({"custom_id": cid, "params": params})
        elif code == "max_tokens":
            params["max_tokens"] *= 2
            requests.append({"custom_id": cid, "params": params})
        elif code == "refusal":
            params["model"] = FALLBACK_MODEL
            requests.append({"custom_id": cid, "params": params})
        elif code == "schema":
            params["messages"] += [
                {"role": "assistant", "content": entry["answer"]},
                {"role": "user", "content": "Validation feedback: your answer did not match the schema. "
                                            "Return corrected JSON only."}]
            requests.append({"custom_id": cid, "params": params})
        elif code == "prompt_too_long":
            for i, part in enumerate(chunk_text(entry["text"], chunk_chars)):
                chunk_params = copy.deepcopy(params)
                chunk_params["messages"] = [{"role": "user", "content": f"<document>\n{part}\n</document>"}]
                requests.append({"custom_id": f"{cid}-c{i}", "params": chunk_params})
        else:
            human.append(cid)
    return requests, human
```
</details>